# UK Exports Predictor

This notebook is the project write-up. It learns from global HS6 goods trade what Flow to expect between any Exporter, Importer and Product, applies that to the UK to estimate **Export Potential** for every Product–Market pair, and ranks positive **Export Gaps** as **Opportunities**.

All reusable logic lives in the `exportspredictor` package, so the notebook and the API compute features identically. The notebook only calls it.

## Parameters

`FIXTURE_MODE = True` (the default) runs everything on a tiny synthetic world in seconds; the test suite executes the notebook this way. With `FIXTURE_MODE = False` (set `EXPORTSPREDICTOR_FIXTURE_MODE=0`, as `scripts/rebuild.py` does), section 1 downloads BACI (once) into the data folder set by `EXPORTSPREDICTOR_DATA_DIR`, section 4 builds the real feature table, and section 5 tunes and trains the real model on the GPU, section 7 trains the test-year models and evaluates them (about an hour in all on an RTX 3080), and section 8 saves the artefacts the API serves.

In [ ]:
import os

# Fixture mode unless EXPORTSPREDICTOR_FIXTURE_MODE=0 (scripts/rebuild.py sets it for the real run).
FIXTURE_MODE = os.environ.get("EXPORTSPREDICTOR_FIXTURE_MODE", "1") != "0"

In [ ]:
import polars as pl

from exportspredictor import artefacts, baci, pipeline, policy
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir


## 1. The Flows: BACI HS12

The training data is CEPII's BACI, HS 2012 revision, release V202601 (ADR 0003): every non-zero Flow between about 226 Exporters and Importers for about 5,200 Products, 2012–2024. BACI reconciles each country's export report with its partners' import reports, so a UK Flow is a blend of what the UK and the Market declared, not HMRC's figure. Values are converted from thousand USD to USD.

This section is the course's **exploratory data analysis** and **data-quality audit**, applied at a scale where pandas would not fit: the download is converted once from CSV to Parquet with compact types (Enum codes for countries and Products, so `010121` keeps its leading zero), and Polars scans it lazily, so only the columns and years a query needs are read. The conversion peaks at about 2.5 GB of memory.

In fixture mode the same summaries run on the fixture world's Flows.

In [ ]:
if FIXTURE_MODE:
    flows = build_world().flows.lazy()
else:
    print(baci.prepare())  # idempotent: skips the download and conversion once done
    flows = baci.scan_flows()
flows.head(5).collect()

### Coverage and size per year

Row counts per year, with the number of Products, Exporters and Importers that trade, total world value and the UK's share of it. BACI records only strictly positive Flows, so a missing row means zero *or* unreported: the model treats absent pairs as zeros, which matters for poorly reporting countries.

In [ ]:
flow_summary = baci.summarise_flows(flows)
flow_summary

In [ ]:
flows.filter(pl.col("value_usd") > 0).select(
    flows=pl.len(),
    products=pl.col("hs6").n_unique(),
    exporters=pl.col("exporter").n_unique(),
    importers=pl.col("importer").n_unique(),
).collect()

On the real data (V202601) there are about 9.0 M Flows in 2012, rising to 11.8 M in 2023, about 142 M in total. 5,199 of the 5,202 HS 2012 Products trade in a typical year, and 226 Exporters appear each year (227 across all years). The UK's share of world goods exports in BACI drifts down from about 2.4–2.6% in 2012–2016 to about 1.9% in 2021–2023.

Flow values are extremely skewed: the mean UK Flow (about $1.3 M) is roughly 100 times its median (about $14k), which is why the model uses a Tweedie loss rather than squared error on raw values (course: skewed variables favour the median).

In [ ]:
uk_latest = flows.filter((pl.col("exporter").cast(pl.String) == baci.UK) & (pl.col("value_usd") > 0))
uk_latest.group_by("year").agg(
    mean_usd=pl.col("value_usd").mean(),
    median_usd=pl.col("value_usd").median(),
    markets=pl.col("importer").n_unique(),
    products=pl.col("hs6").n_unique(),
).sort("year").collect()

### The provisional 2024 data

BACI's latest year is provisional: Comtrade is downloaded in January, before every country has reported, and CEPII revises it in the next release. The table below compares each year with the one before. On the real data, 2024 has about 5.5% fewer Flows than 2023 while world value is almost flat, consistent with late reporters still missing small Flows. UK exports fall about 8% in 2024, which may partly reflect the same incompleteness. 2024 is the Baseline year, so the app labels it provisional, and the evaluation (test year 2023) never relies on it.

In [ ]:
flow_summary.select(
    "year",
    flows_change=pl.col("flows").pct_change(),
    world_value_change=pl.col("world_value_usd").pct_change(),
    uk_value_change=pl.col("uk_value_usd").pct_change(),
).tail(3)

## 2. Gravity and WDI features: coverage

The gravity model of trade (the oldest empirical regularity in trade economics) says Flows grow with the size of both countries and shrink with the distance between them. These are the non-policy fundamentals the model conditions on. This is feature engineering in the course's sense: domain knowledge turned into inputs, built once in the package (`exportspredictor.gravity`) so training and serving share it.

- **Pair features** from CEPII Gravity (V202211): distance between main cities, contiguity, common official language, colonial tie (ever in a colonial or dependency relationship) and common colonizer. They barely change over time, so each pair takes its latest non-null value.
- **Country-year features**: GDP (current USD) and population for every year 2012–2024.

`gravity.build()` downloads what is missing and writes the derived tables; the data rebuild script runs it. It needs BACI's country code file, so it runs after section 1. The cells below run on real data only (`FIXTURE_MODE = False`).

**Country codes.** BACI uses its own numeric codes. All 227 Exporters and Importers in the 2012–2024 Flows map to an ISO3 code, and every one has Gravity pair features. Three mismatches needed a fix:

- BACI 490 "Other Asia, nes" (ISO3 field `S19`) is Taiwan, mapped to `TWN`.
- Saint Barthélemy (`BLM`) and the French Southern and Antarctic Lands (`ATF`) are missing from Gravity. Their pair features come from a stand-in, Guadeloupe and Réunion. Together they carry about $2.6 bn of Flows over 13 years.

**Missing GDP and population: the rule.** For each country and variable:

1. World Bank WDI, the main source.
2. Otherwise the IMF World Economic Outlook. This covers Taiwan (absent from WDI), South Sudan from 2016, Yemen from 2019 and Eritrea to 2019.
3. Otherwise CEPII Gravity's own figures, available up to 2021.
4. Otherwise linear interpolation between the nearest observed years, or the nearest year carried forward or back at the edges. This applies, for example, to Cuba's GDP from 2021 and Syria's from 2023.
5. Otherwise missing. 21 small territories have no GDP in any source, including Anguilla, the Falklands, Gibraltar, North Korea and Saint Helena. Together they account for about 0.1% of world Flow value. XGBoost learns a default direction for missing values at each split, so these rows still train. Imputing a guess would add noise without adding information.

Each value records its source in `gdp_source` / `population_source`, so any figure can be traced. Some pair features are also missing in Gravity: common language for some territories (e.g. Guam, Curaçao), and colonial ties for China–Hong Kong, China–Taiwan and Israel–Palestine. These follow the same rule and stay missing.

In [ ]:
from exportspredictor import gravity
from exportspredictor.paths import data_dir

if not FIXTURE_MODE:
    print(gravity.build())  # downloads once, then rebuilds the derived tables (about 40 s)
    derived = data_dir() / "derived"
    in_flows = pl.concat(
        [flows.select(baci_iso3=pl.col(side).cast(pl.String)) for side in ["exporter", "importer"]]
    ).unique().collect()
    # Only countries that appear in the 2012-2024 Flows matter; BACI's file also lists retired codes.
    country_map = pl.read_parquet(derived / "country_map.parquet").join(in_flows, on="baci_iso3", how="semi")
    country_year = pl.read_parquet(derived / "country_year.parquet").filter(pl.col("iso3").is_in(country_map["iso3"].implode()))
    print("Exporters and Importers in the Flows:", country_map["baci_iso3"].n_unique())
    print("... with no Gravity pair features:", gravity.unresolved(country_map).height)
    print("... with no WDI data:", country_map.filter(~pl.col("in_wdi")).height)
    display(country_map.filter((pl.col("baci_iso3") != pl.col("iso3")) | pl.col("gravity_proxy").is_not_null()))

Where each GDP and population value comes from, and the countries that stay missing:

In [ ]:
if not FIXTURE_MODE:
    display(country_year.group_by("gdp_source").len().sort("len", descending=True))
    display(country_year.group_by("population_source").len().sort("len", descending=True))
    display(country_year.filter(pl.col("gdp_usd").is_null()).group_by("iso3").len().sort("iso3"))

## 3. Policy features: Trade Agreements and Trading Blocs

The policy features are the only columns a Scenario changes, so they must be right for every year. They come from the WTO's Regional Trade Agreements database (RTA-IS), "Export all RTAs" file. It lists every notified agreement with its goods entry-into-force date and **per-member entry and exit dates** (`docs/research/trade-agreement-coverage.md`). CEPII Gravity, the usual source in trade economics, stops in 2021, so it misses UK–Australia, UK–New Zealand and the UK's CPTPP accession.

For each pair and year, we build:

- `trade_agreement`: a goods Trade Agreement is in force between the two countries;
- `customs_union`: the agreement is a Customs Union (WTO type "CU");
- `eu_o/_d`, `cptpp_o/_d`, `efta_o/_d`: Trading Bloc membership of the Exporter and the Importer.

Each rule below follows how the WTO and Gravity code agreements:

- **The 1 July rule** (Gravity's): a membership counts in a year if it began before 1 July and had not ended by then. EU–Japan (1 Feb 2019) counts from 2019, and UK–Australia and UK–NZ (31 May 2023) from 2023. The UK's CPTPP accession (15 Dec 2024) counts only from 2025, so the 2024 Baseline does not include it.
- **A bloc that is a party is expanded to its members at each date.** "EU – United Kingdom" lists only "European Union" as the EU side. Croatia is still covered, because it is an EU member. An "EU – X" agreement links the EU side with X, not X's partners with each other.
- **The UK in 2020:** the UK left the EU (the Trading Bloc) on 31 Jan 2020. It stayed in the EU Customs Union and the EU's agreements until 31 Dec 2020 (the transition period). So for 2020 the UK's `eu_o` is 0, while its `customs_union` with EU members is still 1. The "EU Customs Union only" Scenario relies on this distinction.
- **Partial-scope agreements** (WTO type "PSA", such as the GSTP among developing countries) cut tariffs on short product lists only, so they do not count as Trade Agreements. Gravity's `fta_wto` leaves them out too.

This applies the course's **feature engineering** with explicit **data lineage**. Every flag traces back to a dated record in a primary source. `python -m exportspredictor.policy` rebuilds the tables from scratch; it is a step in `scripts/rebuild.py`.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world carries its own trade_agreement column (section 4).")
else:
    policy_files = policy.build()  # downloads the WTO export once, then rebuilds in seconds
    agreements = pl.read_parquet(policy_files["agreements"])
    trading_blocs = pl.read_parquet(policy_files["trading_blocs"])
    print(agreements.height, "pair-years with a Trade Agreement;", trading_blocs.height, "country-years in a tracked Trading Bloc")

The UK's policy features for a few Markets show the known dates:

- the EU Customs Union until 2020;
- the UK–EU agreement from 2021;
- Australia and New Zealand from 2023;
- no CPTPP membership yet in 2024.

In [ ]:
if not FIXTURE_MODE:
    uk_rows = pl.DataFrame(
        {"exporter": "GBR", "importer": ["FRA", "JPN", "AUS", "NZL", "MEX", "USA"]}
    ).join(pl.DataFrame({"year": list(range(2018, 2025))}, schema={"year": pl.Int32}), how="cross")
    uk_policy = policy.attach_policy(uk_rows, agreements, trading_blocs)
    display(uk_policy.pivot(on="year", index="importer", values="trade_agreement"))
    display(uk_policy.pivot(on="year", index="importer", values="customs_union"))
    display(uk_policy.filter(pl.col("importer") == "FRA").select("year", "eu_o", "cptpp_o", "efta_o"))

### Cross-check against CEPII Gravity, 2012–2021

Gravity V202211 codes `fta_wto` (per pair) and `eu_o/_d` (per country) up to 2021. It uses the same WTO database, with corrections by Thierry Mayer. We compare every pair-year that Gravity has. Most pairs have no agreement in either source, so the plain agreement rate flatters us. We therefore also report `positive_overlap`: of the pair-years that either source codes as in an agreement, the share that both do (the Jaccard index).

In [ ]:
if not FIXTURE_MODE:
    gravity_policy = policy.read_gravity_policy()  # from the gravity step's extract (section 2)
    crosscheck, disagreements = policy.gravity_crosscheck(agreements, trading_blocs, gravity_policy)
    display(crosscheck)

These are the results on the WTO export downloaded on 16 Nov 2026:

- **Trade Agreement:** the agreement rate is 95.5% over 549,900 pair-years, and the positive overlap is 70.6%. Both sources code 58,938 pair-years; 21,426 are ours only and 3,162 are Gravity only.
- **EU membership:** the agreement rate is 99.96%. The one difference is Croatia in 2013. Croatia joined on 1 July 2013, which is not *before* 1 July, so here it counts from 2014. Gravity counts any accession during the year.

Most of the **ours only** pair-years have a clear source:

- **Countries that Gravity never codes in any agreement (14%).** Gravity has no agreement at all for Montenegro, Bermuda or Yemen, and almost none for Liechtenstein. Yet the WTO lists, for example, EU–Montenegro (2008) and the Pan-Arab FTA.
- **Romania before 2018 (5%).** Gravity codes Romania's EU agreements only from 2018, although Romania joined the EU in 2007. This is a gap in Gravity.
- **The African Continental FTA (22%).** The WTO records it in force from 30 May 2019. Gravity does not code it, perhaps because trading under it began only in 2021.
- **The EU's Economic Partnership Agreements with African and Caribbean states (20%):** Eastern and Southern Africa, SADC, CARIFORUM and Ghana. They are notified to the WTO and applied, but Gravity's corrected `fta_wto` leaves most of these pairs out.
- **The rest (39%):** mostly COMESA accessions, SADC, EU–Serbia and EU–Central America. Here Gravity's corrections drop members that the export lists without dates.

Most of the **Gravity only** pair-years also have a clear source:

- **Territories missing from the WTO member lists (17%).** Mayotte, Réunion and other French overseas regions are part of the EU, but the WTO lists only France.
- **Haiti (17%).** Haiti signed the CARIFORUM agreement but has not ratified it. The WTO gives its entry date as 9999.
- **Latin American agreements notified as partial scope under LAIA**, such as Argentina–Bolivia and Argentina–Chile. We exclude these because they are PSAs.

The changes that the Scenarios and the Event Backtest rely on agree in both sources: EU–Japan from 2019, the UK's EU Customs Union to 2020, UK–EU and UK–Japan in 2021. Japan's only disagreements are CPTPP with Canada and New Zealand (in force 30 Dec 2018) and Japan–Mongolia (2016), which Gravity does not code.

UK pairs account for 368 of the disagreeing pair-years: 252 are ours only and 116 Gravity only. They are of the kinds above: Romania, Montenegro and Liechtenstein, the EPA partners, and a year's difference in when EU–Central America started. One more kind appears only for the UK. In 2021, Gravity keeps the UK in EU agreements that the UK did not roll over at the start of 2021, for example with Algeria, Syria, Andorra and the overseas territories. The WTO lists no UK agreement with these partners.

In [ ]:
if not FIXTURE_MODE:
    display(
        disagreements.filter(pl.col("feature") == "trade_agreement")
        .explode("years")
        .group_by("ours", "gravity")
        .agg(pair_years=pl.len(), countries=pl.concat_list("exporter", "importer").explode().n_unique())
    )
    display(disagreements.filter((pl.col("exporter") == "GBR") | (pl.col("importer") == "GBR")).head(20))

## 4. The feature table

Every model input is computed in one place, `exportspredictor.features`, which the notebook (training) and the API (serving and Scenarios) both import, so a feature cannot drift between the two. This is the course's **feature engineering**, built under explicit **leakage** rules.

| Group | Features | What it captures |
|---|---|---|
| Gravity | log GDP and population of both countries, log distance, contiguity, common language, colonial tie, common colonizer | size and closeness of the two countries (section 2) |
| Supply | `log_supply_loo`: the Exporter's exports of the Product to all *other* Importers; `rca_loo`: its revealed comparative advantage in the Product | what the Exporter makes and sells well |
| Demand | `log_demand_loo`: the Importer's imports of the Product from all *other* Exporters; `high_income_share_loo`: the share of those from high-income Exporters | what the Market buys, and from whom ("Romania buys medicines from developed economies") |
| Product | `sector`: the HS2 chapter | Sector-level differences |
| Policy | Trade Agreement, Customs Union, EU, CPTPP and EFTA membership of each side (section 3) | the only columns a Scenario changes; monotone-constrained (ADR 0002) |

### Leakage rules

- **Leave-one-out supply and demand.** A row's own Flow is subtracted from every aggregate it feeds. For a UK row, UK supply of the Product excludes the target Market (ADR 0001). Otherwise the feature would contain the answer, and the out-of-fold model would copy Actual Exports back, shrinking every Export Gap. Revealed comparative advantage also leaves the row's Flow out of the Exporter's and the world's totals.
- **No features from the pair's trade in other Products.** Total UK–Japan trade, for example, is left out on purpose. It is high largely *because* of the policy conditions between the two countries, so it would absorb the policy effect. A Scenario that flips only the policy columns would then understate the Scenario Effect.
- **Training years only.** The clip ranges and the Sector encoder (`FeatureTransform`) are fitted on training years. Test-year data cannot reach them, and serving inputs are clipped to the range seen in training (as tariffs would be, ADR 0002). Aggregates are computed within each year, so a year's features never use a later year's Flows.

The test suite checks each rule (`tests/test_features.py`). It changes the UK–Japan whisky Flow and confirms that this row's supply and demand features stay the same, while the same Product's UK–Canada row changes. It moves the UK's other-Product exports from Japan to Canada and confirms that no UK–Japan whisky feature moves. And it distorts the test years' Flows a thousandfold and confirms that the fitted transform is unchanged.

### Training rows: non-zero Flows plus sampled relevant zeros

BACI records only non-zero Flows, but a model of Export Potential must also learn where trade does *not* happen. The full Exporter × Importer × Product × year grid has about 3.45 bn cells, too many to train on, and most of it is uninformative (Malta not exporting jet engines). So the training rows are:

- **every non-zero Flow**, with weight 1;
- **a random sample of relevant zero cells**, with weight 1/rate. A cell is a relevant zero when the Exporter exports the Product to someone and the Importer imports it from someone that year, but not from each other. These are the zeros that carry information: both sides are active in the Product, and yet no trade happens between them.

On BACI there are about 95 M relevant zero cells a year against 9–12 M non-zero Flows. The default rate of 2% keeps about 1.9 M zeros a year, roughly one for every six non-zero Flows. Each kept zero carries weight 50, so weighted totals stay unbiased: this is the Horvitz–Thompson estimator, which the fixture test checks against an exact count. This goes beyond the course's treatment of class imbalance. Instead of rebalancing classes, we subsample the majority and reweight it, so the Tweedie loss still sees zeros in their true proportion. Whether a cell is kept depends only on a hash of its keys and a seed, so the sample is reproducible and never shifts when other data change. The rate and seed are recorded in the artefact metadata. The rate is fixed at 2% rather than tuned: the hyperparameter search in section 5 is kept small so that the real run stays within a few hours.

### The UK grid for scoring

For scoring, the UK's grid of every Product × Market is built in full for each year, with no sampling: about 5,200 Products × 225 Markets ≈ 1.17 M pairs. About 72% of them have no Actual Exports. Each pair's Export Potential comes from its out-of-fold model.

On real data, `features.build()` writes one file per year for the training rows and one for the UK grid (`derived/features_V202601/`). It builds 2012–2024 in about 90 s, with a measured peak memory of 7.9 GB, well inside 32 GB, because each year is built from that year's Flows alone.

In [ ]:
from exportspredictor import features

if FIXTURE_MODE:
    feature_world = build_world()
    feature_table = features.build_features(
        features.training_rows(feature_world.flows, rate=0.5), feature_world.flows, feature_world.sources
    ).lazy()
else:
    feature_dir = features.build()  # about 90 s for 2012-2024, peak memory about 8 GB
    feature_table = pl.scan_parquet(feature_dir / "training_*.parquet")

is_zero = pl.col("value_usd") == 0
feature_table.group_by("year").agg(
    rows=pl.len(),
    sampled_zeros=is_zero.sum(),
    relevant_zeros_estimated=pl.col("weight").filter(is_zero).sum(),
).sort("year").collect()

On real data, 2023 has 11.8 M non-zero Flows and 1.89 M sampled zeros (13.6 M rows). The weights estimate about 94.5 M relevant zero cells. Across 2012–2024 there are about 165 M training rows, about 3.3 GB of Parquet.

The features of one UK row, UK whisky to Japan in the latest year:

In [ ]:
feature_table.filter(
    (pl.col("exporter") == "GBR") & (pl.col("importer") == "JPN") & (pl.col("hs6").cast(pl.String) == "220830")
).sort("year").tail(1).collect().transpose(include_header=True, column_names=["value"])

## 5. Training: tuning, out-of-fold fold models and the Baseline

The model is a single gradient-boosted tree ensemble (XGBoost, ADR 0002), trained on the GPU (`device="cuda"`). It extends the course's tree-ensemble material in four ways:

- **Tweedie loss.** Flows are non-negative, often zero and extremely skewed, so squared error on raw values would chase a few huge Flows. The Tweedie objective models a log-mean with a variance that grows with the mean, which suits exactly that (course: skewed targets and loss choice).
- **Monotone constraints** (+1 on every policy feature, ADR 0002): adding a Trade Agreement, Customs Union or Trading Bloc membership never lowers Export Potential. The model *associates* an agreement with higher Flows; it does not show cause.
- **Out-of-fold cross-fitting for the UK** (ADR 0001, beyond the course's k-fold cross-validation). UK Product–Market pairs are split into 5 folds by a hash of the pair, so every year of a pair shares a fold. Fold model *k* trains on all non-UK rows plus the UK pairs outside fold *k*, and scores only fold *k*. A pair's Export Potential therefore never comes from a model that saw its Actual Exports, so the Export Gap is not shrunk by memorisation. `tests/test_folds.py` checks this by distorting one fold's UK labels a thousandfold: that fold's scores do not move at all, while every other fold's do.
- **Training data that does not fit in memory.** The ~150 M training rows of 2012–2023 are about 16 GB as floats. They are streamed to XGBoost one year at a time through a data iterator into a `QuantileDMatrix`, which stores only the binned features. Even binned, all of them fill the RTX 3080's 10 GB once training starts, so the real run keeps a hash sample of 30% of *non-UK* rows, reweighted by 1/0.3 (the same Horvitz–Thompson idea as the sampled zeros in section 4). Every UK row is kept.

### The time split and the search

The years are split as the course recommends for data with a time order (course: train/validation/test splits and leakage):

| Role | Years | Used for |
|---|---|---|
| training (search) | 2012–2021 | fit each candidate configuration |
| validation | 2022 | early stopping and choosing the configuration |
| test | 2023 | the evaluation (next work); never read here |
| refit | 2012–2023 | the final fold models that score the Baseline |
| Baseline | 2024 (provisional) | Export Potential, Actual Exports and Export Gap for every UK pair |

The search is deliberately small, five configurations: tree depth 6, 8 or 10, and Tweedie variance power 1.3, 1.5 or 1.7 at depth 8. Each is trained with early stopping on 2022 (course: early stopping as regularisation). Candidates with different variance powers optimise different losses, so they are all compared on one fixed metric, the weighted Tweedie negative log-likelihood at power 1.5. The feature transform (clip ranges and Sector encoder) is fitted on the search's training years for the search, and on 2012–2023 for the refit, never on a year it is scored on.

In [ ]:
if FIXTURE_MODE:
    world = build_world()
    bundle = pipeline.build_artefacts(world, source="fixture")
else:
    bundle = pipeline.build_real_artefacts()  # about an hour on an RTX 3080, evaluation included
{key: bundle.metadata[key] for key in ("source", "device", "params", "rounds", "tuning_seconds", "training_seconds", "non_uk_sampling_rate")}

The search results on the 2022 validation year, best first (lower is better):

In [ ]:
tuning = pl.DataFrame(bundle.metadata["tuning"]).unnest("config")
tuning

On the real data (run of 17 Nov 2026, artefact version `20261117T002435Z`), the search took 8.3 minutes on the RTX 3080 and the refit of the five fold models 20.5 minutes, about 30 minutes in all. Both times are recorded in the artefact metadata.

| Depth | Variance power | Rounds (early stopping) | Validation Tweedie NLL @1.5 |
|---|---|---|---|
| 8 | 1.3 | 1,491 | **341.5** |
| 10 | 1.5 | 420 | 345.6 |
| 8 | 1.5 | 875 | 346.4 |
| 6 | 1.5 | 1,481 | 349.3 |
| 8 | 1.7 | 400 | 353.5 |

The differences are small (about 3% between best and worst), so the model is not sensitive to these choices. A lower variance power wins: it treats large Flows as less noisy, so the model works harder to fit them, and large Flows dominate the Export Gaps Advisors read. The winning configuration stopped at 1,491 of the 1,500 allowed rounds, so a longer budget might gain a little more. We keep the cap so that the run stays short. Whether the tuned model beats a naive baseline on the untouched 2023 test year is the job of the evaluation section.

### The Baseline table

The refit fold models score the full 2024 UK grid, every traded Product × every Market, each pair by the fold model that held it out. The Baseline table compares Export Potential with Actual Exports; a positive Export Gap is an Opportunity. The artefacts also store, for the Scenarios and explanations to come, each pair's fold and model inputs, and the UK's Actual Exports for 2019–2024.

In [ ]:
baseline = bundle.baseline.join(bundle.products, on="hs6", how="left").join(
    bundle.markets.rename({"name": "market"}), on="iso3", how="left"
)
baseline.select(
    pairs=pl.len(),
    products=pl.col("hs6").n_unique(),
    markets=pl.col("iso3").n_unique(),
    pairs_with_exports=(pl.col("actual_exports_usd") > 0).sum(),
    actual_exports_usd=pl.col("actual_exports_usd").sum(),
    export_potential_usd=pl.col("export_potential_usd").sum(),
    opportunities=(pl.col("export_gap_usd") > 0).sum(),
)

### Sanity check: the top Opportunities

A model can score well and still rank nonsense, so we read the largest Export Gaps and ask whether a trade economist would find them plausible.

In [ ]:
baseline.sort("export_gap_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"
).head(15)

One Product an Advisor would know well, whisky (`220830`): the Markets with the largest Export Gaps, and the ones where the UK already exceeds its Export Potential.

In [ ]:
whisky = baseline.filter(pl.col("hs6") == "220830").sort("export_gap_usd", descending=True)
whisky_columns = ["market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"]
display(whisky.select(whisky_columns).head(8))
whisky.select(whisky_columns).tail(5)

On the real data, the Baseline covers all 1,169,550 UK Product × Market pairs of 2024 (5,198 Products × 225 Markets), of which 306,102 have Actual Exports, together $407 bn. Read the results with these points in mind:

- **Export Potential totals $692 bn, and 95% of pairs have a positive Export Gap.** This is what a conditional mean does: a pair with no exports but plausible fundamentals gets a small positive Export Potential, and there are 860k such pairs. So individual small gaps are noise, and only the *size* ranking is informative. On pairs with exports, the Spearman rank correlation between Export Potential and Actual Exports is 0.69, so the model reproduces the shape of UK trade without copying it (the pairs were held out).
- **The top Opportunities are plausible.** They are concentrated in the UK's known strengths: medicines to China and the Netherlands; aircraft parts to Italy, Singapore, China and France; jet engines and their parts to Hong Kong, France and Poland; cars and tractors to Germany. These are large Markets for Products the UK makes well, where comparable Exporters sell more.
- **Some top gaps have explanations the model cannot see.** Crude oil to Ireland and gas to France and China depend on pipelines, refineries and long-term contracts, not on the fundamentals in our features.
- **Sanctions are invisible to the model.** For whisky, Hong Kong leads, then the **Russian Federation**: UK whisky exports there fell from $211 M in 2021 to $19 M in 2022 under the sanctions (the history table shows it), but no feature records sanctions. So an Advisor would read this row as a Barrier the model does not know about, not as an Opportunity. This is a known limitation for the Model Evaluation page.
- **Whisky's negative gaps are where the UK already over-performs**: the USA, Singapore, Taiwan, France and India, its long-established whisky Markets.

As always, these are associations: the model says that comparable Exporters sell more of these Products to these Markets, not that the UK would if it tried.

## 6. Scenarios and Blocked Opportunities

A Scenario asks the trained model a what-if question: copy the 2024 Baseline rows of the affected Markets, change **only the policy columns**, and re-predict each pair with the same fold model that scored its Baseline (ADR 0001). The Scenario Effect is the difference. `exportspredictor.scenarios` does this for the notebook and the API alike, and checks on every run that no non-policy column moved (a leakage guard: a Scenario that also shifted, say, demand would report the demand change as a policy effect).

This extends two course concepts:

- **Ensembles: gradient boosting.** The fold models carry a +1 monotone constraint on every policy feature (ADR 0002), so switching a Trade Agreement on can never lower a prediction. A property-based test checks this over the fixture rows with random policy settings. Without the constraint, a tree could learn a spurious dip and a Scenario could show a negative effect from adding an agreement.
- **Correlation, multicollinearity and causation.** Flipping one feature and re-predicting is the same move as the course's counterfactual fairness test, but the answer is still learned from correlations in past trade. So we say "the model associates the CPTPP with +X%", never "joining CPTPP causes +X%". The Event Backtest (#21) is what tests these effects against a real agreement.

The four Trading Bloc presets (wayfinder #8) change only the rows of the bloc's member Markets in the Baseline year:

| Preset | Policy columns set on member Markets' rows |
|---|---|
| EU re-join | `trade_agreement`, `customs_union`, `eu_o` (as for any pair of EU members in training) |
| EU Customs Union only | `customs_union` |
| CPTPP | `trade_agreement`, `cptpp_o`, with accession treated as complete |
| Leave | for each bloc the UK is in: its `<bloc>_o`, `trade_agreement` and `customs_union` switched off |

Because a membership counts in a year only if it began before 1 July, the 2024 Baseline has no UK CPTPP membership (accession took effect from December 2024), so the CPTPP preset is a live prediction that 2025 data can check. In 2024 the UK is in no tracked Trading Bloc, so Leave changes nothing, and the API says so.

In [ ]:
from exportspredictor import scenarios

preset_results = {
    name: scenarios.run(bundle.baseline_features, bundle.baseline, bundle.fold_models, scenarios.Scenario(preset=name))
    for name in scenarios.PRESETS
}
pl.DataFrame(
    [
        {
            "preset": scenarios.PRESETS[name].title,
            "pairs_changed": result.pairs.height,
            **result.total(),
            "blocked_opportunities": result.blocked_opportunities().height,
        }
        for name, result in preset_results.items()
    ]
)

On the real artefact (version `20261117T003531Z`), on CPU:

| Preset | Pairs changed | Scenario Effect | Share of Baseline | Blocked Opportunities (≥20%, ≥$1M) |
|---|---|---|---|---|
| EU re-join | 140,346 (27 Markets) | +$151.3bn | +44.7% | 13,458 |
| EU Customs Union only | 140,346 | +$58.4bn | +17.3% | 3,285 |
| CPTPP | 57,178 (11 Markets) | +$5.7bn | +8.1% | 422 |
| Leave | 0 | 0 | | 0 |

Each bloc Scenario returns in about a second through the API. The EU figures are large because the model associates intra-EU membership with much denser trade than a free-trade agreement alone; they describe what the model has learned from EU members' trade, not what re-joining would bring. The CPTPP effect is smaller: the UK already has agreements with 9 of the 11 members, so for those nine only the CPTPP flag changes, and Brunei and Malaysia also gain an agreement. The CPTPP effect per Market:

In [ ]:
preset_results["cptpp"].by_market().join(bundle.markets.select("iso3", "name"), on="iso3", how="left")

### Precomputed Blocked Opportunity values

The rankings can also be ordered by **Blocked Opportunity value**: for every pair with no Trade Agreement in 2024, the Scenario Effect of adding one, re-predicted by the pair's fold model when the artefacts are built (`scenarios.blocked_effects`). It is 0 where an agreement already exists. A pair is a Blocked Opportunity when that effect is at least 20% of its Baseline Export Potential and at least $1M. The largest values:

In [ ]:
baseline.with_columns(
    blocked_opportunity=scenarios.is_blocked("blocked_effect_usd", "export_potential_usd")
).sort("blocked_effect_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "blocked_effect_usd", "blocked_opportunity"
).head(10)

## 7. Evaluation: accuracy on the 2023 test year

The course's **time split** is the honest way to evaluate a model that will be used on later data: fit on the past, test on a year the model never saw. Here the years are train 2012–2021, validate 2022 (the hyperparameter search of section 5) and test 2023. The Baseline models of section 5 were refit on 2012–2023, so they cannot be tested on 2023. The evaluation therefore trains five **test-year fold models** on 2012–2022 with the configuration and round count chosen on 2022, and scores 2023 with them. Nothing about 2023 shaped tuning or training: the pipeline asserts that the test year is not among the tuning years, and a test (`test_the_test_year_is_untouched_during_tuning`) multiplies every 2023 Flow by 1,000 and checks that the search results do not move. This is the course's **leakage** rule applied to model selection, not only to preprocessing.

Following the course's **baselines and metrics** advice (report the naive baseline first), every metric is computed for the model and for a **naive baseline**: the Exporter's share of the Product's world exports times the Importer's imports of that Product. It is the "no fundamentals" prediction a trade analyst would make on a spreadsheet, and like the model's features it leaves the Flow itself out.

The course distinguishes the **loss** a model minimises from the **metric** people read. We report one of each kind and one that goes beyond the notes:

| Metric | Kind | Why |
|---|---|---|
| Tweedie deviance (power 1.5, weighted) | the training loss | measures the fit on every row, zeros included; sampled zeros count by their weight |
| RMSE of log(1 + v), non-zero Flows | the course's RMSE, on a log scale | the typical error in orders of magnitude; raw-dollar RMSE would be ruled by a handful of huge Flows |
| Spearman rank correlation within each Product | a ranking metric (beyond the notes) | Advisors read the model as a ranking of Markets for one Product, so we score exactly that, averaged over Products |

Two scopes are evaluated. **All Exporters** scores every 2023 training row (every non-zero Flow plus the sampled zero cells): UK rows out-of-fold, other rows by the mean of the five fold models. **UK out-of-fold** scores the UK's full 2023 Product × Market grid, each pair by the fold model that never saw it (ADR 0001), which is how the Baseline is produced. As in section 5, the models train on every UK row and a 30% reweighted sample of non-UK rows on real data. The fixture keeps every row.

The headline is the within-Product rank correlation for UK pairs, phrased for an Advisor:

In [ ]:
from exportspredictor import evaluation

report = bundle.evaluation
print(report["headline"]["text"])
{key: report[key] for key in ("test_year", "test_training_years", "validation_year", "non_uk_sampling_rate", "evaluation_seconds")}

Every metric, model against naive baseline, for both scopes:

In [ ]:
pl.DataFrame([{"scope": scope, **result["overall"]} for scope, result in report["accuracy"].items()]).drop("group", strict=False)

### Where the model is weak: Sectors and regions

The breakdowns double as the **bias review** the course asks for in its *govern* theme. Here bias is about countries and Sectors, not people: BACI's data are weaker where countries report late or carelessly, and the model may be weaker for those Markets. Each table is sorted weakest first, by the model's within-Product rank correlation for UK pairs.

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_sector"]).head(10)

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_region"])

### Stability of the 2024 Baseline

BACI's 2024 data are provisional, so we check that the Baseline's rankings are not an artefact of one noisy year. The Baseline models also score the 2023 UK grid, and we compare the two years' rankings within each Product, and the overlap of the top 1,000 Opportunities. This is also the course's **drift** check in miniature: when BACI releases a new year, the same comparison shows how far the rankings moved.

In [ ]:
print(report["stability"]["text"])
report["stability"]

### Known limitations

These are shown on the app's Model Evaluation page, so that Advisors can caveat their briefings. The last one came out of section 5's sanity check of the top Opportunities.

In [ ]:
pl.DataFrame(evaluation.LIMITATIONS)

REAL_RESULTS_PLACEHOLDER

## 8. Saving the artefacts

The API serves exactly what was trained here: a versioned artefact folder with the five fold models, the Baseline table, the Baseline's model inputs and folds, the Actual Exports history, the Product and Market lookups, the evaluation report of section 7 (`evaluation.json`), and a metadata file (sources, sampling rates, the search, the chosen parameters, device and timings).

In [ ]:
saved_to = artefacts.save(bundle, artefacts_dir())
saved_to